# Importing Header and SoilPrep

In [2]:
%run Header.ipynb
%run SoilPrep.ipynb

C:\Users\swetasingh\anaconda3\Lib\site-packages\nbformat\__init__.py:93: MissingIDFieldWarning: Code cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  validate(nb)
C:\Users\swetasingh\anaconda3\Lib\site-packages\nbformat\__init__.py:93: MissingIDFieldWarning: Code cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  validate(nb)


ModuleNotFoundError: No module named 'Header'

ModuleNotFoundError: No module named 'Header'

# Step 0: Setting up decision parameters (Data Tree)

In [ ]:
# 0. Available smoothing filter types: savgol1 and savgol2 ------------------------ (0)
sg_filters = ['sg2']

# 0. Available window lengths for the smoothing filter ---------------------------- (0)
window_lengths = [0, 1, 11, 21, 31, 41, 51, 71, 91]

# 1. Available machine learning regression models --------------------------------- (1)
ml_methods = ['ridge', 'svr', 'plsr', 'cubist', 'gbrt']

# 2. Names of target variables in the dataframe ----------------------------------- (2)
# Get the current working directory
current_directory = os.getcwd()
# Get the name of the current directory
current_folder_name = os.path.basename(current_directory)
target_names = [current_folder_name]

# 3. Available preprocessing for Target data -------------------------------------- (3)
prepare_target = ['none']
# 4. Available preprocessing for Spectral data ------------------------------------ (4)
prepare_spec = ['none', 'cr', 'log', 'fod', 'fod_cr', 'fod_log']

# 5. Resampling bands available for spectra --------------------------------------- (5) 
nbands_sampling = [5,10,15,20,25,30,35, 40, 45, 50, 60, 70, 80, 90, 100]

# Setting colours for different targets        --------------------------------------
clr = ['#F4A460', '#8B7355', '#A52A2A']

# Colour scheme definition
kado = '#8B7355'
mati = '#A52A2A'
balu = '#F4A460'

In [ ]:
target_names[0]

In [ ]:
MetaData = {'sg_filters' : sg_filters, 'window_lengths' : window_lengths, 'prepare_spec' : prepare_spec, \
            'nbands_sampling' : nbands_sampling, 'target_names' : target_names, 'prepare_target' : prepare_target,\
            'ml_methods' : ml_methods, 'clr' : clr}

with open ('meta_data.pickle', 'wb') as file:
    pickle.dump(MetaData, file)

## Step 0- Data Imputation

In [ ]:
legacy_spec = pd.read_csv('../spectra.csv')
legacy_tar = pd.read_csv('../targets.csv')

legacy_spec.rename(columns= {'sample':'id'}, inplace=True)
legacy_tar.rename(columns = {'SOIL': 'id', 'CLAI':'Clay', 'SILT': 'Silt', 'SAND': 'Sand', 'OC': 'TOC', 'CACO3':'CaCO3'}, inplace = True)

In [ ]:
legacy_spec = legacy_spec.set_index('id', drop=True)
legacy_tar = legacy_tar.set_index('id', drop=True)

In [ ]:
legacy_spectra = legacy_spec.iloc[:, 51::].copy()
legacy_spectra.head(2)

In [ ]:
legacy_tar = legacy_tar[['Clay', 'Silt', 'Sand', 'TOC', 'CaCO3']]
legacy_tar.head(2)

In [ ]:
# Missing ID's are taken care by setting the index to ID and obtaining inner join.


legacy_fr = pd.merge(legacy_tar, legacy_spectra, on = 'id', how = 'inner')

legacy_fr.reset_index(inplace = True)
legacy_fr .head(2)

In [ ]:
print('Missing:', legacy_fr.isnull().sum().sum())   # alternate to previous (inferred from next command)
print(legacy_fr.isnull().sum()) # isnull applies to df but isnan applies only to ndarray

legacy_fr.dropna(axis=0, inplace = True)

In [ ]:
spectra = legacy_fr.iloc[:, 6::]
spectra.head(2)

In [ ]:
target = legacy_fr.copy()

In [ ]:
spectra.shape

In [ ]:
target.shape

In [ ]:
# Ensure the indices are aligned
spectra.reset_index(drop=True, inplace=True)
target.reset_index(drop=True, inplace=True)

In [ ]:
assert spectra.index.equals(target.index), "Indices are not aligned between features and target."

# Step 1a: Obtaining Spectra (Noise and Outliers removal)

In [ ]:
for i in range (0,10,1):
    spectra.iloc[i,1:].plot()

# Step 1b: Obtaining Targets (Isolating as series)

In [ ]:
clr = ['#F4A460', '#8B7355', '#A52A2A']

def isolate_targets(target, target_names):
    T=[]
    for i in range (0,len(target_names)):
        T.append(target[target_names[i]])
    return(T)
    
T = isolate_targets(target,target_names)

# Step 1c: Spectra Preprocessing (Smooth, FOD/Contin/Log , and Resample)

In [ ]:
#halt here

## Savgol smoothing (order 2)

In [ ]:
# -------------- Smoothed Spectra  spec2 (savgol order 2)  -----------

spec2 = {}
for i in window_lengths:
    spec2[i] = filt_sg(spectra, i, 'sg2')

smth_spec = sgsmooth (spectra, 3)    

In [ ]:
fod_spec = fod(smth_spec)

for i in range (0,5,1):
    fod_spec.iloc[i,:].plot()


## Continuum Removal

In [ ]:
cr_spec = continuum_removed(spec2[51])

for i in range (0,5,1):
    cr_spec.iloc[i,:].plot()
    

## log(1/R) Transformation

In [ ]:
log_spec = ((1/spec2[51]).apply(np.log)).copy()
#log_spec.head(5)

for i in range (0,5,1):
    log_spec.iloc[i,:].plot()   

## Resampling (n_bands)

### 1. Sampled Original (sampled_spec)

### 2. Sampled Continuum Removed  (sampled_cr)

### 3. Sampled Log (sampled_log)

In [ ]:
sampled_spec = {}
sampled_cr = {}
sampled_log = {}
for n in nbands_sampling:
    sampled_spec[n] = resample_spectra (spec2[51], n)
    sampled_cr[n] = resample_spectra (cr_spec, n)
    sampled_log[n] = resample_spectra (log_spec, n)

### 4.  FOD of sampled spectra (fod_sampled)

### 5. FOD of sampled_cr (fod_sampledcr)

### 6. FOD of sampled_log (fod_sampledlog)

In [ ]:
fod_sampled = {}
fod_cr = {}
fod_log = {}
for n in nbands_sampling:
    fod_sampled[n] = fod (sampled_spec[n])
    fod_cr[n] = fod (sampled_cr[n])
    fod_log[n] = fod (sampled_log[n])

## Visualizing Processed Spectrum (variable samples)

In [ ]:
(row, col) = spectra.shape
(row, col)

In [ ]:
def plot_spec (sample, process):
    (row, col) = spectra.shape
    x1 = spec2[51].iloc[sample,:]
    x1.plot()
    if process == 'cr':
        x2 = cr_spec.iloc[sample,:]
        x2.plot()
    elif process == 'log':
        x3 = log_spec.iloc[sample,:]/3
        x3.plot()
    else:
        x4 = fod_spec.iloc[sample,:]*100
        x4.plot()
        
    plt.ylim([-0.6, 0.9])

ipywidgets.interact(plot_spec, sample = (0, row, 1), process = prepare_spec)

## Correlation between wavelengths and Targets

In [ ]:
def find_rpval (spectra, tar):
    (r, c) = spectra.shape
    
    r_val = spectra.iloc[[0], :].copy()
    p_val = spectra.iloc[[0], :].copy()
    
    for j in range(0, c):
        #print("Length of tar:", len(tar))
        #print("Length of spectra.iloc[:, j]:", len(spectra.iloc[:, j]))

        r_val.iloc[0,j], p_val.iloc[0,j] = stats.pearsonr(tar, spectra.iloc[:, j])
    
    return(r_val, p_val)


In [ ]:
plt.style.use(['science','notebook','grid'])

def plot_corr (target, prepare, n_bands):
    
    i = target_names.index(target)    
    
    if  prepare == 'none':
        r_val, p_val = find_rpval (sampled_spec[n_bands], T[i])
        r_val.iloc[0,:].plot(color = clr[i])
    elif  prepare == 'cr':
        r_cr, p_cr = find_rpval (sampled_cr[n_bands], T[i])
        r_cr.iloc[0,:].plot(color = clr[i])
    elif prepare == 'log':
        r_log, p_log = find_rpval (sampled_log[n_bands], T[i])
        r_log.iloc[0,:].plot(color = clr[i])
    elif prepare == 'fod_spec':    
        r_fod, p_fod = find_rpval (fod_sampled[n_bands], T[i])
        r_fod.iloc[0,:].plot(color = clr[i]) 
    elif prepare == 'fod_cr':    
        r_sfodcr, p_sfodcr = find_rpval (fod_cr[n_bands], T[i])
        r_sfodcr.iloc[0,:].plot(color = clr[i]) 
    else:   
        r_sfodlog, p_sfodlog = find_rpval (fod_log[n_bands], T[i])
        r_sfodlog.iloc[0,:].plot(color = clr[i]) 
    
    plt.ylim([-0.9, 0.9])
    plt.show()

ipywidgets.interact(plot_corr, target = target_names, prepare = prepare_spec, n_bands = nbands_sampling)



In [ ]:
Data = {'spectra' : spec2[51], 'T' : T, 'spec2': spec2, 'smth_spec' : smth_spec, 'fod_spec' : fod_spec,  \
       'cr_spec' : cr_spec,  'log_spec' : log_spec, 'sampled_spec' : sampled_spec, 'sampled_cr' : sampled_cr, \
        'fod_sampled' : fod_sampled, 'sampled_log' : sampled_log, 'fod_cr' : fod_cr, 'fod_log' : fod_log}

In [ ]:
with open ('data.pickle', 'wb') as file:
    pickle.dump(Data, file)